# Validating the `spey-roofit` plug-in

This notebook validates [`spey-roofit`](https://github.com/SpeysideHEP/spey-roofit),
the [`RooFit`](https://root.cern/manual/roofit/) backend for
[`spey`](https://spey.readthedocs.io), on three independent fronts:

1. **Against `spey`'s own likelihoods.** Two `HistFactory` workspaces are built that are
   mathematically identical to `default.uncorrelated_background` and
   `default.correlated_background`, and every quantity `spey` can compute is compared
   between the two implementations.
2. **Against `pyhf`.** The same model is written as a `pyhf` workspace. `pyhf` uses the
   same Asimov conventions as `RooFit`, so the agreement is expected to be exact for the
   likelihood, the Asimov data *and* the exclusion limits.
3. **Against the published ATLAS monojet analysis**
   ([arXiv:2102.10874](https://arxiv.org/abs/2102.10874),
   [HEPData ins1847779](https://www.hepdata.net/record/ins1847779)). The interpreter has
   to reproduce the published yields of all 78 regions, and the profile likelihood has to
   recover the signal strength stored in the published post-fit workspace.

Run it in the environment described by `environment.yml`.

> **Runtime.** Everything up to and including section 4 runs in a couple of minutes.
> The two exclusion limits at the very end are full profile-likelihood scans of a
> 150-parameter model and take a few minutes each; the outputs are stored in this
> notebook, so it can be read without re-running it.


In [1]:
import json
import os
import sys

import numpy as np
import spey

import spey_roofit
from spey_roofit import WorkspaceInterpreter, load_workspace
from spey_roofit.data import WorkspaceData

REPOSITORY = (
    os.path.dirname(os.getcwd())
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)
sys.path.insert(0, os.path.join(REPOSITORY, "tests"))
ATLAS = os.path.join(REPOSITORY, "atlas-exot-2018-06")

print("spey        ", spey.__version__)
print("spey-roofit ", spey_roofit.__version__)
print("ROOT        ", spey_roofit.manager.version)
print("backends    ", [b for b in spey.AvailableBackends() if "roofit" in b])

Spey - WARNING: An unstable version of Spey (0.2.8-b1) is being used. Latest stable version is 0.2.7.
spey         0.2.8-b1
spey-roofit  0.0.1
ROOT         6.36.04
backends     ['root.roofit']


---
## 1. A `RooFit` twin of `spey`'s uncorrelated-background likelihood

`spey`'s `default.uncorrelated_background` backend implements

$$
\mathcal{L}(\mu,\theta)=\prod_i \mathrm{Pois}\!\left(n_i\,\middle|\,\mu s_i+b_i+\theta_i\sigma_i\right)
\cdot\prod_i \mathrm{Gauss}\!\left(0\,\middle|\,\theta_i,1\right).
$$

The same model is expressed in `HistFactory` by giving each bin its own `HistoSys` whose
up and down templates are $b_i\pm\sigma_i$: the piecewise-linear interpolation is then
exact and the constraint is the usual unit Gaussian on $\alpha$. The helper that builds
it lives in `tests/toy_models.py`.

In [3]:
from toy_models import correlated_background_workspace, uncorrelated_background_workspace

WORKDIR = os.path.join(REPOSITORY, "notebooks", "_generated")
os.makedirs(WORKDIR, exist_ok=True)

background = [50.0, 60.0, 30.0]
signal = [12.0, 11.0, 5.0]
uncertainty = [3.0, 5.0, 2.0]
observed = [51.0, 48.0, 32.0]

uncorrelated_path = uncorrelated_background_workspace(
    WORKDIR, background, signal, observed, uncertainty
)

### What the interpreter sees

The interpreter is the user-facing view of a workspace: it lists the regions, their
binning, the samples and their yields, without requiring any knowledge of `RooFit`.

In [4]:
interpreter = WorkspaceInterpreter(uncorrelated_path)
print(interpreter.summary())

RooFit workspace 'combined'
  channels            : 1
  bins                : 3
  parameters of interest : mu
  nuisance parameters : 3
  global observables  : 3

  [__unknown__] ch (3 bins)
      observed   : [51, 48, 32]
      background : [50, 60, 30]
      signal     : [12, 11, 5]
        - bkg: [50, 60, 30]
        - sig (signal): [12, 11, 5]
RooFit workspace 'combined'
  channels            : 1
  bins                : 3
  parameters of interest : mu
  nuisance parameters : 3
  global observables  : 3

  [__unknown__] ch (3 bins)
      observed   : [51, 48, 32]
      background : [50, 60, 30]
      signal     : [12, 11, 5]
        - bkg: [50, 60, 30]
        - sig (signal): [12, 11, 5]


In [5]:
roofit = spey.get_backend("root.roofit")(
    workspace=uncorrelated_path, analysis="roofit-toy"
)
reference = spey.get_backend("default.uncorrelated_background")(
    signal_yields=signal,
    background_yields=background,
    data=observed,
    absolute_uncertainties=uncertainty,
    analysis="spey-toy",
)

config = roofit.backend.config()
print("parameters :", config.parameter_names)
print("poi index  :", config.poi_index)
print(
    "minimum poi:", config.minimum_poi, "vs spey", reference.backend.config().minimum_poi
)

parameters : ['mu', 'alpha_theta0', 'alpha_theta1', 'alpha_theta2']
poi index  : 0
minimum poi: -4.166666666666667 vs spey -4.166666666666667


### Log-likelihood

The first and most important check: the two backends must return the *same number* for
$\log\mathcal{L}(\mu,\theta)$, not just the same likelihood ratios. `RooFit` normalises its
negative log-likelihood differently from `spey` — it adds data-dependent but
parameter-independent terms such as the category normalisation of a `RooSimultaneous` —
and the plug-in calibrates that offset away once per dataset.

In [6]:
ours = roofit.backend.get_logpdf_func()
theirs = reference.backend.get_logpdf_func()

rng = np.random.default_rng(2024)
rows = []
for _ in range(8):
    pars = np.concatenate([[rng.uniform(-2.0, 3.0)], rng.normal(0.0, 0.8, 3)])
    rows.append((pars, ours(pars), theirs(pars)))

print(f"{'mu':>8s} {'roofit logL':>18s} {'spey logL':>18s} {'difference':>14s}")
for pars, a, b in rows:
    print(f"{pars[0]:8.3f} {a:18.12f} {b:18.12f} {a - b:14.2e}")
print()
print("max |difference| =", max(abs(a - b) for _, a, b in rows))

      mu        roofit logL          spey logL     difference
   1.379   -23.271320091408   -23.271320091407      -2.84e-14
   2.979   -36.926006579485   -36.926006579485       4.26e-14
  -0.202   -13.319795056460   -13.319795056460      -2.84e-14
  -1.473   -16.287664473832   -16.287664473832       0.00e+00
   2.878   -31.104925608414   -31.104925608414       1.42e-14
  -0.968   -16.028035687426   -16.028035687426      -2.84e-14
   2.036   -24.335757729709   -24.335757729709       3.20e-14
   0.336   -13.537772303338   -13.537772303338       0.00e+00

max |difference| = 4.263256414560601e-14


### Everything `spey` derives from the likelihood

`likelihood`, `maximize_likelihood` and the observed exclusion are compared directly.

In [7]:
def compare(name, fn, tolerance=1e-6):
    a, b = fn(roofit), fn(reference)
    a, b = float(np.ravel(a)[0]), float(np.ravel(b)[0])
    delta = abs(a - b) / max(abs(b), 1e-12)
    flag = "ok" if delta < tolerance else "differs"
    print(f"{name:34s} {a:16.10f} {b:16.10f} {delta:10.2e}  {flag}")


print(f"{'quantity':34s} {'spey-roofit':>16s} {'spey default':>16s} {'rel. diff':>10s}")
compare("likelihood(mu=0)", lambda m: m.likelihood(0.0))
compare("likelihood(mu=1)", lambda m: m.likelihood(1.0))
compare("likelihood(mu=2)", lambda m: m.likelihood(2.0))
compare("maximize_likelihood -> muhat", lambda m: m.maximize_likelihood()[0], 1e-3)
compare("maximize_likelihood -> nll", lambda m: m.maximize_likelihood()[1])
compare("asimov_likelihood(mu=1)", lambda m: m.asimov_likelihood(1.0), 1e-1)
compare("maximize_asimov_likelihood", lambda m: m.maximize_asimov_likelihood()[1], 1e-1)
compare("exclusion_confidence_level", lambda m: m.exclusion_confidence_level()[0], 1e-2)
compare("poi_upper_limit", lambda m: m.poi_upper_limit(), 1e-1)

quantity                                spey-roofit     spey default  rel. diff
likelihood(mu=0)                      12.1379528821    12.1379528175   5.32e-09  ok
likelihood(mu=1)                      15.3485728768    15.3485728968   1.31e-09  ok
likelihood(mu=2)                      22.0472740180    22.0472740222   1.90e-10  ok
maximize_likelihood -> muhat          -0.2386704561    -0.2387357267   2.73e-04  ok
maximize_likelihood -> nll            12.0017105887    12.0017103005   2.40e-08  ok
asimov_likelihood(mu=1)               13.3238188618    13.3238246489   4.34e-07  ok
maximize_asimov_likelihood            11.1969570301    11.1969591683   1.91e-07  ok
exclusion_confidence_level             0.9838873970     0.9838874178   2.11e-08  ok
poi_upper_limit                        0.7904797341     0.7904795164   2.75e-07  ok


In [21]:
print("Asimov data")
print("  spey-roofit :", np.round(roofit.generate_asimov_data(), 6))
print("  spey default:", np.round(reference.generate_asimov_data(), 6))

Asimov data
  spey-roofit : [ 5.0151384e+01  5.6310481e+01  3.0232837e+01  5.0461000e-02
 -7.3790400e-01  1.1641900e-01]
  spey default: [ 5.0151258e+01  5.6309521e+01  3.0233577e+01  5.0419000e-02
 -7.3809600e-01  1.1678800e-01]


---
## 2. A `RooFit` twin of the correlated-background likelihood

`default.correlated_background` constrains the background with a correlated Gaussian,
$\theta\sim\mathcal{N}(0,\rho)$ and $\lambda_i=\mu s_i+b_i+\sigma_i\theta_i$.
Factorising $\Sigma=LL^{T}$ and using the columns of $L$ as `HistoSys` templates gives the
*same* model written in an uncorrelated basis, $\lambda_i=\mu s_i+b_i+\sum_k L_{ik}z_k$
with $z\sim\mathcal{N}(0,\mathbb{1})$.

Changing basis multiplies the likelihood by a constant Jacobian, so the two
log-likelihoods differ by exactly $\tfrac12\log\det\rho$ — and every likelihood ratio,
and therefore every inference result, is unchanged.

In [9]:
covariance = np.array([[9.0, 2.0, 1.0], [2.0, 25.0, 3.0], [1.0, 3.0, 4.0]])
correlated_path = correlated_background_workspace(
    WORKDIR, background, signal, observed, covariance
)

factor = np.linalg.cholesky(covariance)
scale = np.diag(np.sqrt(np.diag(covariance)))
correlation = np.linalg.inv(scale) @ covariance @ np.linalg.inv(scale)
constant = 0.5 * np.log(np.linalg.det(correlation))

roofit_corr = spey.get_backend("root.roofit")(
    workspace=correlated_path, analysis="roofit-corr"
)
reference_corr = spey.get_backend("default.correlated_background")(
    signal_yields=signal,
    background_yields=background,
    data=observed,
    covariance_matrix=covariance,
    analysis="spey-corr",
)

ours = roofit_corr.backend.get_logpdf_func()
theirs = reference_corr.backend.get_logpdf_func()

print("predicted constant offset = 0.5 log|rho| =", constant)
print()
rng = np.random.default_rng(7)
for _ in range(5):
    poi = rng.uniform(-1.0, 2.0)
    latent = rng.normal(0.0, 1.0, 3)
    theta = np.linalg.inv(scale) @ (factor @ latent)
    delta = ours(np.concatenate([[poi], latent])) - theirs(np.concatenate([[poi], theta]))
    print(
        f"  mu={poi:7.4f}   logL difference = {delta:.12f}   residual = {delta - constant:.2e}"
    )

predicted constant offset = 0.5 log|rho| = -0.06518090893162179

  mu= 0.8753   logL difference = -0.065180908932   residual = -1.30e-15
  mu=-0.0995   logL difference = -0.065180908932   residual = 4.72e-16
  mu= 1.3912   logL difference = -0.065180908932   residual = -2.97e-14
  mu=-0.2354   logL difference = -0.065180908932   residual = 4.72e-16
  mu= 1.9865   logL difference = -0.065180908932   residual = -1.20e-14


In [10]:
print("muhat  spey-roofit:", roofit_corr.maximize_likelihood()[0])
print("muhat  spey default:", reference_corr.maximize_likelihood()[0])

muhat  spey-roofit: -0.23386253782443253
muhat  spey default: -0.2338396136004408


---
## 3. Cross-check against `pyhf`

`pyhf` implements the same `HistFactory` statistical model, including the convention that
the auxiliary data of an Asimov dataset follow the nuisance parameters. The agreement with
`spey-roofit` should therefore be exact for the likelihood, the Asimov data and
$\mathrm{CL}_s$.

In [11]:
import pyhf

pyhf.set_backend("numpy", "scipy")

channels, observations = [], []
for index, (b, s, u, n) in enumerate(zip(background, signal, uncertainty, observed)):
    channels.append(
        {
            "name": f"b{index}",
            "samples": [
                {
                    "name": "bkg",
                    "data": [b],
                    "modifiers": [
                        {
                            "name": f"theta{index}",
                            "type": "histosys",
                            "data": {"hi_data": [b + u], "lo_data": [b - u]},
                        }
                    ],
                },
                {
                    "name": "sig",
                    "data": [s],
                    "modifiers": [{"name": "mu", "type": "normfactor", "data": None}],
                },
            ],
        }
    )
    observations.append({"name": f"b{index}", "data": [n]})

pyhf_workspace = pyhf.Workspace(
    {
        "channels": channels,
        "observations": observations,
        "measurements": [{"name": "m", "config": {"poi": "mu", "parameters": []}}],
        "version": "1.0.0",
    }
)
pyhf_model = pyhf_workspace.model()
pyhf_data = pyhf_workspace.data(pyhf_model)

order = pyhf_model.config.par_order
positions = [order.index(name) for name in ("mu", "theta0", "theta1", "theta2")]


def to_pyhf(pars):
    translated = np.zeros(pyhf_model.config.npars)
    for source, target in enumerate(positions):
        translated[target] = pars[source]
    return translated


ours = roofit.backend.get_logpdf_func()
print(f"{'mu':>8s} {'spey-roofit':>18s} {'pyhf':>18s} {'difference':>12s}")
rng = np.random.default_rng(5)
for _ in range(6):
    pars = np.concatenate([[rng.uniform(-1.0, 2.0)], rng.normal(0.0, 0.6, 3)])
    a = ours(pars)
    b = float(pyhf_model.logpdf(to_pyhf(pars), pyhf_data)[0])
    print(f"{pars[0]:8.3f} {a:18.12f} {b:18.12f} {a - b:12.2e}")

      mu        spey-roofit               pyhf   difference
   1.415   -19.005944789536   -19.005944789536     0.00e+00
  -0.838   -13.624390443469   -13.624390443469     1.78e-15
  -0.854   -13.922211470312   -13.922211470312    -2.84e-14
   0.305   -14.732287528948   -14.732287528948     0.00e+00
   0.177   -12.869209703360   -12.869209703360     2.84e-14
   0.667   -14.880534353204   -14.880534353204     0.00e+00


In [12]:
pars = [1.0, 0.3, -0.4, 0.2]
print("expected data")
print("  spey-roofit:", np.round(roofit.backend.expected_data(pars), 10))
print("  pyhf       :", np.round(pyhf_model.expected_data(to_pyhf(pars)), 10))
print()
cls_pyhf = float(pyhf.infer.hypotest(1.0, pyhf_data, pyhf_model, test_stat="qtilde"))
print("CLs at mu = 1")
print("  spey-roofit (1 - CLs):", roofit.exclusion_confidence_level()[0])
print("  pyhf        (1 - CLs):", 1.0 - cls_pyhf)

expected data
  spey-roofit: [62.9 69.  35.4  0.3 -0.4  0.2]
  pyhf       : [62.9 69.  35.4  0.3 -0.4  0.2]

CLs at mu = 1
  spey-roofit (1 - CLs): 0.9838873970379719
  pyhf        (1 - CLs): 0.9838872017114197


---
## 4. The published ATLAS monojet workspaces

`atlas-exot-2018-06/` contains the two `RooFit` workspaces published with the ATLAS
full Run-2 monojet search:

| file | content |
| --- | --- |
| `bOnlyFit_postFit.root` | background-only fit configuration, post-fit nuisance parameters |
| `sPlusBFit_postFit.root` | signal-plus-background fit for the `DMA_1_2000_gq0p25` benchmark |

Neither requires the user to know anything about `RooFit`: the interpreter exposes the
region structure directly.

In [13]:
atlas = WorkspaceInterpreter(os.path.join(ATLAS, "bOnlyFit_postFit.root"))
print(atlas)
print()
print("signal regions :", atlas.guess_signal_regions())
print()
print("control regions:", len(atlas.guess_control_regions()))
print("bins per region:", sorted(set(atlas.bin_map.values())))

WorkspaceInterpreter(channels=78, bins=78, poi=none, injected=[])

signal regions : ['SR_200_cuts', 'SR_250_cuts', 'SR_300_cuts', 'SR_350_cuts', 'SR_400_cuts', 'SR_500_cuts', 'SR_600_cuts', 'SR_700_cuts', 'SR_800_cuts', 'SR_900_cuts', 'SR_1000_cuts', 'SR_1100_cuts', 'SR_1200_cuts']

control regions: 65
bins per region: [1]


In [14]:
print(atlas.summary(max_channels=2))

RooFit workspace 'w'
  channels            : 78
  bins                : 78
  parameters of interest : none (background only)
  nuisance parameters : 146
  global observables  : 175

  [SR] SR_200_cuts (1 bin)
      observed   : [1.792e+06]
      background : [1.783e+06]
        - NCB_200: [3513]
        - multijet_200: [1.86e+04]
        - Znunu_200: [9.87e+05]
        - Wmunu_200: [1.74e+05]
        - Wenu_200: [1.166e+05]
        - Wtaunu_200: [3.651e+05]
        - Zmumu_200: [6383]
        - Ztautau_200: [4985]
        - ttbar_200: [4.735e+04]
        - singletop_200: [1.938e+04]
        - diboson_200: [2.041e+04]
        - ZjetsVBF_200: [1.103e+04]
        - WjetsVBF_200: [9128]
  [CR] CR1mu0b_200_cuts (1 bin)
      observed   : [7.427e+05]
      background : [7.416e+05]
        - Wmunu_200: [6.313e+05]
        - Wtaunu_200: [3.852e+04]
        - Zmumu_200: [1.364e+04]
        - ttbar_200: [2.498e+04]
        - singletop_200: [1.254e+04]
        - diboson_200: [9627]
        - Wjet

### Reproducing the published yields

HEPData tables 2–7 give the observed counts and the post-fit background prediction of
every region. They are stored in `atlas-exot-2018-06/hepdata/`. The interpreter has to
reproduce both, to the precision the tables are quoted with.

In [15]:
CUTS = [200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200]
REGIONS = {
    "SR": "Histogram_SR",
    "CR1mu0b": "Histogram_CR1mu0b",
    "CR1e0b": "Histogram_CR1e0b",
    "CR1L1b": "Histogram_CR1L1b",
    "CR2mu": "Histogram_CR2mu",
    "CR2e": "Histogram_CR2e",
}

observed_data = atlas.observed_data
background_yields = atlas.expected_background_yields

worst = 0.0
for region, table in REGIONS.items():
    with open(
        os.path.join(ATLAS, "hepdata", f"{table}.json"), encoding="utf-8"
    ) as handle:
        rows = json.load(handle)["values"]
    if region == "SR":
        print(
            f"{'pT recoil':>10s} {'data (HEPData)':>16s} {'data (RooFit)':>16s}"
            f" {'bkg (HEPData)':>16s} {'bkg (RooFit)':>16s} {'rel':>9s}"
        )
    for row, cut in zip(rows, CUTS):
        channel = f"{region}_{cut}_cuts"
        published_data = float(row["y"][0]["value"])
        published_background = float(row["y"][1]["value"])
        our_data = observed_data[channel][0]
        our_background = background_yields[channel][0]
        assert our_data == published_data, channel
        relative = abs(our_background - published_background) / published_background
        worst = max(worst, relative)
        if region == "SR":
            print(
                f"{cut:>10d} {published_data:16.1f} {our_data:16.1f}"
                f" {published_background:16.1f} {our_background:16.1f} {relative:9.1e}"
            )
print()
print(f"largest relative deviation over all {6 * len(CUTS)} regions: {worst:.2e}")
print("(the published numbers are rounded to five significant digits)")

 pT recoil   data (HEPData)    data (RooFit)    bkg (HEPData)     bkg (RooFit)       rel
       200        1791624.0        1791624.0        1783400.0        1783436.4   2.0e-05
       250         752328.0         752328.0         752990.0         752989.6   4.9e-07
       300         313912.0         313912.0         313990.0         313988.8   3.9e-06
       350         141036.0         141036.0         140100.0         140104.8   3.4e-05
       400         102888.0         102888.0         101560.0         101561.6   1.6e-05
       500          29458.0          29458.0          29244.0          29243.6   1.2e-05
       600          10203.0          10203.0          10003.0          10002.7   3.3e-05
       700           3986.0           3986.0           3872.7           3872.7   7.0e-07
       800           1663.0           1663.0           1644.8           1644.8   3.8e-06
       900            738.0            738.0            754.2            754.2   5.1e-06
      1000           

### Recovering the published signal strength

`sPlusBFit_postFit.root` stores the result of the ATLAS fit: the parameter of interest
`mu_SIG` sits at its fitted value. Maximising the likelihood through `spey` must return
that same value — which exercises the whole chain at once: yield evaluation, constraint
terms, data-vector layout and the `spey` optimiser.

In [16]:
splusb = spey.get_backend("root.roofit")(
    workspace=os.path.join(ATLAS, "sPlusBFit_postFit.root"),
    analysis="atlas-exot-2018-06",
)
published_mu_hat = splusb.backend.model.nominal_parameters[0]
mu_hat, nll = splusb.maximize_likelihood()

print("number of free parameters :", splusb.backend.config().npar)
print("mu_hat stored in workspace:", published_mu_hat)
print("mu_hat from spey          :", mu_hat)
print("relative difference       :", abs(mu_hat - published_mu_hat) / published_mu_hat)

number of free parameters : 151
mu_hat stored in workspace: 0.12684446636721747
mu_hat from spey          : 0.12684459376507368
relative difference       : 1.0043627432672985e-06


### Cross-checking the background model against the `pyhf` likelihood

The same HEPData record also publishes the full likelihood in the `HistFactory` JSON
format. Evaluating the expected yields of all 78 regions with `pyhf` and with
`spey-roofit` is an end-to-end test of the yield extraction on a realistic model with
150 nuisance parameters.

The archive is not shipped with this repository. `atlas-exot-2018-06/download.sh`
fetches it along with everything else this notebook needs; set `ATLAS_PYHF_JSON` if
you keep it somewhere else.

In [17]:
PYHF_JSON = os.environ.get(
    "ATLAS_PYHF_JSON", os.path.join(ATLAS, "bkgonly_workspace.json")
)

if not os.path.exists(PYHF_JSON):
    print(f"{PYHF_JSON} not found - skipping the pyhf cross-check.")
else:
    with open(PYHF_JSON, encoding="utf-8") as handle:
        atlas_spec = json.load(handle)
    atlas_model = pyhf.Workspace(atlas_spec).model(
        modifier_settings={
            "normsys": {"interpcode": "code4"},
            "histosys": {"interpcode": "code4p"},
        }
    )
    roofit_model = WorkspaceData(
        load_workspace(os.path.join(ATLAS, "bOnlyFit_postFit.root")), poi_name="Lumi"
    )

    pars = np.array(atlas_model.config.suggested_init())
    pars[atlas_model.config.poi_index] = 0.0
    expected = atlas_model.expected_data(pars, include_auxdata=False)
    cursor, per_channel = 0, {}
    for name in atlas_model.config.channels:
        nbins = atlas_model.config.channel_nbins[name]
        per_channel[name] = expected[cursor : cursor + nbins]
        cursor += nbins
    pyhf_yields = np.concatenate([per_channel[c] for c in roofit_model.channel_names])

    nominal = [
        0.0 if name.startswith("alpha_") else 1.0 for name in roofit_model.parameter_names
    ]
    roofit_yields = roofit_model.expected_data(nominal)[: roofit_model.nbins]

    deviation = np.abs(pyhf_yields - roofit_yields) / np.abs(pyhf_yields)
    print(f"{'region':22s} {'pyhf':>16s} {'spey-roofit':>16s} {'rel':>10s}")
    for name, a, b, d in zip(
        roofit_model.channel_names, pyhf_yields, roofit_yields, deviation
    ):
        if name.startswith("SR"):
            print(f"{name:22s} {a:16.4f} {b:16.4f} {d:10.1e}")
    print()
    print("largest relative deviation over all 78 regions:", deviation.max())

region                             pyhf      spey-roofit        rel
SR_200_cuts                1569592.6689     1569592.6689    0.0e+00
SR_250_cuts                 665928.6002      665928.6002    0.0e+00
SR_300_cuts                 277806.1623      277806.1623    0.0e+00
SR_350_cuts                 124126.8753      124126.8753    0.0e+00
SR_400_cuts                  90134.2886       90134.2886    0.0e+00
SR_500_cuts                  26005.1299       26005.1299    0.0e+00
SR_600_cuts                   8900.9001        8900.9001    0.0e+00
SR_700_cuts                   3447.1427        3447.1427    0.0e+00
SR_800_cuts                   1462.5200        1462.5200    0.0e+00
SR_900_cuts                    669.4870         669.4870    0.0e+00
SR_1000_cuts                   318.0752         318.0752    0.0e+00
SR_1100_cuts                   161.0121         161.0121    0.0e+00
SR_1200_cuts                   192.0405         192.0405    0.0e+00

largest relative deviation over all 78 regions:

---
## 5. Injecting your own signal

The point of the interpreter is that a user with a new model only needs to know the
region names and their binning — `bin_map` tells them exactly that — and can then hand
`spey` a signal without ever touching `RooFit`.

Below, the published signal template is extracted from the signal-plus-background
workspace, injected into the *background-only* workspace, and the two are compared.

In [18]:
published = WorkspaceData(load_workspace(os.path.join(ATLAS, "sPlusBFit_postFit.root")))
nominal = published.nominal_parameters
template = np.array(
    published.expected_data(np.concatenate([[1.0], nominal[1:]]))
) - np.array(published.expected_data(np.concatenate([[0.0], nominal[1:]])))

signal_yields = {
    channel: [float(template[index])]
    for index, channel in enumerate(published.channel_names)
    if template[index] > 0.0
}
for channel, value in signal_yields.items():
    print(f"  {channel:16s} {value[0]:10.2f}")

  SR_200_cuts         1351.29
  SR_250_cuts         1041.62
  SR_300_cuts          766.13
  SR_350_cuts          548.74
  SR_400_cuts          681.70
  SR_500_cuts          370.21
  SR_600_cuts          211.50
  SR_700_cuts          125.61
  SR_800_cuts           78.53
  SR_900_cuts           47.70
  SR_1000_cuts          28.96
  SR_1100_cuts          18.62
  SR_1200_cuts          34.18


In [19]:
injected = spey.get_backend("root.roofit")(
    workspace=os.path.join(ATLAS, "bOnlyFit_postFit.root"),
    signal_yields=signal_yields,
    analysis="atlas-injected",
)
model = injected.backend.model
print("parameter of interest:", model.poi.GetName())
print("free parameters      :", model.npar)
print("is_alive             :", injected.is_alive)

zero = np.array(model.expected_data([0.0] + list(model.nominal_parameters[1:])))
one = np.array(model.expected_data([1.0] + list(model.nominal_parameters[1:])))
recovered = (one - zero)[: model.nbins]

print()
print(f"{'region':16s} {'requested':>12s} {'recovered':>12s}")
for index, channel in enumerate(model.channel_names):
    if channel in signal_yields:
        print(f"{channel:16s} {signal_yields[channel][0]:12.4f} {recovered[index]:12.4f}")

parameter of interest: mu
free parameters      : 147
is_alive             : True

region              requested    recovered
SR_200_cuts         1351.2946    1351.2946
SR_250_cuts         1041.6185    1041.6185
SR_300_cuts          766.1266     766.1266
SR_350_cuts          548.7405     548.7405
SR_400_cuts          681.7040     681.7040
SR_500_cuts          370.2099     370.2099
SR_600_cuts          211.5005     211.5005
SR_700_cuts          125.6093     125.6093
SR_800_cuts           78.5300      78.5300
SR_900_cuts           47.7045      47.7045
SR_1000_cuts          28.9629      28.9629
SR_1100_cuts          18.6165      18.6165
SR_1200_cuts          34.1795      34.1795


### Full inference on the injected model

Everything `spey` offers is now available. The two workspaces are not identical — the
published one also carries the signal theory systematics, which the injected signal does
not — so the results are close but not equal.

Each of the two limits below takes a few minutes: `RooFit` offers no analytic
gradient, so `spey` differentiates the 150-parameter likelihood numerically.


In [20]:
import time

start = time.time()
print("1 - CLs (injected) :", injected.exclusion_confidence_level()[0])
print(f"   ({time.time() - start:.0f} s)")
start = time.time()
print("1 - CLs (published):", splusb.exclusion_confidence_level()[0])
print(f"   ({time.time() - start:.0f} s)")

1 - CLs (injected) : 0.9803218777674108
   (187 s)
1 - CLs (published): 0.9837091243464943
   (63 s)
